In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Importing Required Libraries
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
#from tensorflow.keras.applications import EfficientNetB0
#from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
#Splitting Input Data into 3 parts that are training,validation,testing
input_folder='/content/drive/MyDrive/Agricultural-crops'
import splitfolders
split_ratio = (0.8, 0.1, 0.1)
splitfolders.ratio(
    input_folder,
    output='crop/dataset',
    seed=500,
    ratio=split_ratio,
    group_prefix=None,
)

Copying files: 829 files [06:06,  2.26 files/s]


In [ ]:
# Define the parameters
img_size = (224, 224)
batch_size = 32

# Data augmentation for training data
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
# Data augmentation for test data (only rescaling)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
# Data augmentation for validation data (only rescaling)
valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [ ]:
# Upgrade pip
!pip install --upgrade pip

# Install required libraries
!pip install split-folders
!pip install tensorflow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
# Create data generators
train_data = train_datagen.flow_from_directory(
    '/content/crop/dataset/train',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    '/content/crop/dataset/test',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

valid_data =valid_datagen.flow_from_directory(
    '/content/crop/dataset/val',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

Found 652 images belonging to 30 classes.
Found 105 images belonging to 30 classes.
Found 72 images belonging to 30 classes.


In [ ]:
#Model Loading
from keras.applications.resnet import ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Freeze the convolutional base
base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# Model Building
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(30, activation='softmax')
])

In [ ]:
# Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Train the model
model.fit(train_data, epochs=50, validation_data=valid_data)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 146s 7s/step - accuracy: 0.0939 - loss: 3.6409 - val_accuracy: 0.3611 - val_loss: 2.7437
Epoch 2/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 131s 6s/step - accuracy: 0.2249 - loss: 2.8171 - val_accuracy: 0.4167 - val_loss: 2.1715
Epoch 3/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 128s 6s/step - accuracy: 0.3051 - loss: 2.3368 - val_accuracy: 0.5833 - val_loss: 1.7408
Epoch 4/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 129s 6s/step - accuracy: 0.3776 - loss: 2.1307 - val_accuracy: 0.6528 - val_loss: 1.4212
Epoch 5/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 127s 6s/step - accuracy: 0.4959 - loss: 1.8006 - val_accuracy: 0.5972 - val_loss: 1.3091
Epoch 6/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 138s 7s/step - accuracy: 0.5512 - loss: 1.6372 - val_accuracy: 0.6667 - val_loss: 1.1308
Epoch 7/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 128s 6s/step - accuracy: 0.5792 - loss: 1.4687 - val_accuracy: 0.7361 - val_loss: 1.0285
Epoch 8/50
21/21 ━━━━━━━━━━━━━━━━━━━━ 144s 6s/step - accuracy: 0.6085 - loss: 1.2852 - val_accuracy: 0.7361 - v

In [ ]:
# Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(test_data)

print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 4s/step - accuracy: 0.8213 - loss: 0.8851
Test Accuracy: 84.76%


In [ ]:
class_names = {0: 'Cherry', 1: 'Coffee-plant', 2: 'Cucumber', 3: 'Fox_nut(Makhana)', 4: 'Lemon', 5: 'Olive-tree',
               6: 'Pearl_millet(bajra)', 7: 'Tobacco-plant', 8: 'almond', 9: 'banana', 10: 'cardamom', 11: 'chilli',
               12: 'clove', 13: 'coconut', 14: 'cotton', 15: 'gram', 16: 'jowar', 17: 'jute', 18: 'maize',
               19: 'mustard-oil', 20: 'papaya', 21: 'pineapple', 22: 'rice', 23: 'soyabean', 24: 'sugarcane',
               25: 'sunflower', 26: 'tea', 27: 'tomato', 28: 'vigna-radiati(Mung)', 29: 'wheat'}

In [ ]:
#Model Prediction
import cv2
import numpy as np
def predict_img(image,model):
    test_img=cv2.imread(image)
    test_img=cv2.resize(test_img,(224,224))
    test_img=np.expand_dims(test_img, axis=0)
    result=model.predict(test_img)
    r=np.argmax(result)
    print(class_names[r])

In [ ]:
#Saving Model
model.save('Crop_CLF_V1.keras')

In [ ]:
print(tf.__version__)

2.19.0


In [ ]:
# INPUT_FOLDER = "/content/drive/MyDrive/MINOR- 3/Agricultural-crops"
# OUTPUT_FOLDER = "/content/Dataset"  # Colab working directory

In [ ]:
import splitfolders

# Split into train/val/test
splitfolders.ratio(
    INPUT_FOLDER,
    output=OUTPUT_FOLDER,
    seed=500,
    ratio=(0.8, 0.1, 0.1)
)

train_dir = f"{OUTPUT_FOLDER}/train"
val_dir   = f"{OUTPUT_FOLDER}/val"
test_dir  = f"{OUTPUT_FOLDER}/test"

Copying files: 829 files [00:05, 155.29 files/s]


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)
valid_data = valid_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)
test_data = test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)


Found 652 images belonging to 30 classes.
Found 72 images belonging to 30 classes.
Found 105 images belonging to 30 classes.


In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze backbone

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_data.num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30)             │         3,870 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,853,854 (91.00 MB)

 Trainable params: 266,142 (1.02 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
import pickle
import os

# Create imagePrediction folder if it doesn't exist

save_dir = 'C:/Users/DHARMENDRA/Desktop/MINOR/ML/AgroLens/backend/imagePrediction'
os.makedirs(save_dir, exist_ok=True)

# Save the model in pickle format
model_path = os.path.join(save_dir, 'crop_prediction_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# Also save class_names for later use
class_names_path = os.path.join(save_dir, 'class_names.pkl')
with open(class_names_path, 'wb') as f:
    pickle.dump(class_names, f)

print(f"✅ Model saved to: {model_path}")
print(f"✅ Class names saved to: {class_names_path}")